In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [3]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

In [4]:
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=2):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.enc4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(512, 1024)

        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(1024, 512)  # 512 (bottleneck) + 512 (enc4)

        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(512, 256)   # 256 + 256

        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 128)   # 128 + 128

        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)    # 64 + 64

        self.out = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Энкодер
        e1 = self.enc1(x)          # 64
        p1 = self.pool1(e1)        # 64

        e2 = self.enc2(p1)         # 128
        p2 = self.pool2(e2)        # 128

        e3 = self.enc3(p2)         # 256
        p3 = self.pool3(e3)        # 256

        e4 = self.enc4(p3)         # 512
        p4 = self.pool4(e4)        # 512

        b = self.bottleneck(p4)    # 1024

        # Декодер
        d4 = self.up4(b)           # 512
        d4 = torch.cat([d4, e4], dim=1)  # 1024
        d4 = self.dec4(d4)         # 512

        d3 = self.up3(d4)          # 256
        d3 = torch.cat([d3, e3], dim=1)  # 512
        d3 = self.dec3(d3)         # 256

        d2 = self.up2(d3)          # 128
        d2 = torch.cat([d2, e2], dim=1)  # 256
        d2 = self.dec2(d2)         # 128

        d1 = self.up1(d2)          # 64
        d1 = torch.cat([d1, e1], dim=1)  # 128
        d1 = self.dec1(d1)         # 64

        out = self.out(d1)         # out_channels
        return out

In [6]:
class WeightedBCELoss(nn.Module):
    def __init__(self, weight_map=None):
        super().__init__()
        self.weight_map = weight_map  # карта весов (H, W)

    def forward(self, pred, target):
        # pred: (batch, 1, H, W) — после сигмоиды
        # target: (batch, 1, H, W) — бинарная маска
        bce = F.binary_cross_entropy(pred, target, reduction='none')
        if self.weight_map is not None:
            bce = bce * self.weight_map.to(pred.device)
        return bce.mean()